# Instagram CSV → SQLite pipeline walkthrough

This notebook documents the project workflow. Place the six expected CSV files in `../instagram details/` when running from this notebook directory, or adjust `DATA_DIR` below. The repository screenshot and charts are conceptual templates until the real dataset is run.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_DIR / 'instagram details'
DB_FILE = PROJECT_DIR / 'insta_lite.db'
print('Project directory:', PROJECT_DIR)
print('Data directory exists:', DATA_DIR.exists())

## 1. Inspect source CSV files

This cell lists files and previews each file if the expected data folder is populated.

In [ ]:
csv_files = sorted(DATA_DIR.glob('*.csv')) if DATA_DIR.exists() else []
print('CSV files found:', [p.name for p in csv_files])
for path in csv_files:
    print(f'\\n--- {path.name} ---')
    display(pd.read_csv(path).head())

## 2. Run the pipeline

Run the script from the project root after placing the expected CSV files in the data folder.

In [ ]:
import subprocess, sys
script_path = PROJECT_DIR / 'instagram_pipeline.py'
if not csv_files:
    print('Add the source CSV files before running the pipeline.')
else:
    result = subprocess.run([sys.executable, str(script_path)], cwd=PROJECT_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Pipeline exited with status {result.returncode}')

## 3. Validate the SQLite database

If the database exists, inspect table names and row counts.

In [ ]:
if DB_FILE.exists():
    with sqlite3.connect(DB_FILE) as conn:
        table_names = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
        display(table_names)
        for table in table_names['name']:
            count = pd.read_sql_query(f'SELECT COUNT(*) AS row_count FROM "{table}"', conn).iloc[0, 0]
            print(f'{table}: {count:,} rows')
else:
    print('Database not found yet. Run the pipeline after adding CSV files.')

## 4. Interaction summary chart

The chart below is generated from real database counts when the database is available.

In [ ]:
if DB_FILE.exists():
    queries = {
        'Likes': "SELECT COUNT(*) FROM interactions WHERE interaction_type='like'",
        'Comments': "SELECT COUNT(*) FROM interactions WHERE interaction_type='comment'",
        'Follows': "SELECT COUNT(*) FROM interactions WHERE interaction_type='follow'",
    }
    with sqlite3.connect(DB_FILE) as conn:
        counts = {name: conn.execute(query).fetchone()[0] for name, query in queries.items()}
    display(pd.DataFrame({'interaction_type': list(counts), 'count': list(counts.values())}))
    plt.figure(figsize=(7,4))
    plt.bar(counts.keys(), counts.values())
    plt.title('Interactions by type')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Run the pipeline to generate data-driven charts.')

## Notes and limitations

- Confirm column names against your exports before using this project with a different dataset.
- Review date parsing failures and missing values.
- The SQLite writer in this starter project infers schemas via pandas; enforce foreign keys explicitly if database-level integrity is required.
- Do not publish private Instagram exports or identifying information.